# Projet 7

## Mission : Intégrez et optimisez le système MLOps

L'objectif de cette partie sera de créer une interface client/modèle avec une API REST permettant de faire une prédiction à l'aide du modèle entrainé.

Le client pourra, via une requête HTTPS, envoyer de nouvelles données clients formatées selon les features du dataset d’entraînement et recevoir en retour les probabilités de solvabilité.

Workflow :

1- création de l'API REST à l'aide de la lib FastAPI dans un fichier api.py 

2- définition de la fonction de prédiction qui appelle le pipeline du modèle entrainé dans un endpoint de l'API

3- test en local le fonctionnement de l'API

4- on met en production l'API : 
    - création d'un conteneur docker pour définir l'environnement d'éxécution de l'API
    - test du docker en local
    - déploiement du docker sur un serveur cloud à l'aide de la lib Render
    - test du docker sur le cloud

5- création d'une interface graphique de l'API avec Streamlite

6- intégration continue de l'api en production 

## 1- API locale

### 1-1 Requêtes http

In [1]:
# import des librairies
import pandas as pd
import numpy as np
import requests

In [2]:
# définition de l'url de l'API
url = "http://127.0.0.1:8000/predict"

In [3]:
# 1- on va récupérer un échantillon de données
data = pd.read_csv("../data_test.csv")

In [9]:
# 2- on va sélectionner un échantillon de 3 clients
data_sample = data.sample(3, random_state=42)

In [10]:
data_sample

,EXT_SOURCE_2,EXT_SOURCE_3,CREDITCARD_CREDIT_UTILIZATION_MEAN,CREDITCARD_CNT_DRAWINGS_ATM_CURRENT_MEAN,CREDITCARD_CNT_DRAWINGS_CURRENT_MAX,EXT_SOURCE_1,CREDITCARD_CREDIT_UTILIZATION_MAX,DAYS_CREDIT_mean,CREDITCARD_CNT_DRAWINGS_CURRENT_MEAN,CREDIT_ACTIVE_Closed_mean,...,FLAG_DOCUMENT_6,NAME_HOUSING_TYPE_House / apartment,FLAG_WORK_PHONE,CNT_PAYMENT_mean,CNT_PAYMENT_sum,OCCUPATION_TYPE_Low-skill Laborers,PAYMENT_DIFF_SUM,APARTMENTS_MODE,NAME_FAMILY_STATUS_Single / not married,AMT_PAYMENT_MIN
15071,0.771890,NaN,NaN,NaN,NaN,0.637061,NaN,NaN,NaN,NaN,...,0.0,1.0,0.0,24.000000,24.0,0.0,0.00,0.0378,0.0,16490.25
10157,0.672038,0.554947,NaN,NaN,NaN,NaN,NaN,-537.375,NaN,0.875,...,1.0,1.0,0.0,37.333333,336.0,0.0,-250996.50,0.2363,1.0,7454.97
36313,0.630215,0.619528,0.0,NaN,0.0,0.789471,0.0,-1755.250,0.0,0.250,...,0.0,1.0,0.0,9.750000,78.0,0.0,-214788.78,NaN,0.0,4046.13


In [112]:
type (data_sample)

pandas.core.frame.DataFrame

In [11]:
# Il faut supprimer les valeurs nulles
data_input = data_sample.replace({np.nan: None, np.inf: None, -np.inf: None})

Nos données se présentent sous forme d'un DataFrame Pandas, les formats nativement acceptés par les APIs sont les formats json.
Il existe plusieurs techniques pour formater les données en json, la plus courante consiste à utiliser JSON orienté 'records'. Cette technique permet de transformer chaque ligne du dataframe en un dictionnaire et toutes les lignes, s'il y en a plusieurs, en liste de dictionnaires.

In [12]:
# 3- on va convertir cet échantillon en json
data_json = data_input.to_dict(orient="records")

In [115]:
data_json

[{'EXT_SOURCE_2': 0.771890322148218,
  'EXT_SOURCE_3': None,
  'CREDITCARD_CREDIT_UTILIZATION_MEAN': None,
  'CREDITCARD_CNT_DRAWINGS_ATM_CURRENT_MEAN': None,
  'CREDITCARD_CNT_DRAWINGS_CURRENT_MAX': None,
  'EXT_SOURCE_1': 0.6370605089336666,
  'CREDITCARD_CREDIT_UTILIZATION_MAX': None,
  'DAYS_CREDIT_mean': None,
  'CREDITCARD_CNT_DRAWINGS_CURRENT_MEAN': None,
  'CREDIT_ACTIVE_Closed_mean': None,
  'DAYS_BIRTH': -11348.0,
  'CREDIT_ACTIVE_Active_mean': None,
  'DAYS_CREDIT_min': None,
  'CREDITCARD_AMT_INST_MIN_REGULARITY_MEAN': None,
  'CREDITCARD_CREDIT_UTILIZATION_STD': None,
  'DAYS_CREDIT_UPDATE_mean': None,
  'CREDITCARD_CNT_DRAWINGS_ATM_CURRENT_MAX': None,
  'REGION_RATING_CLIENT_W_CITY': 1.0,
  'REGION_RATING_CLIENT': 1.0,
  'NAME_INCOME_TYPE_Working': 1.0,
  'NAME_EDUCATION_TYPE_Higher education': 1.0,
  'DAYS_ENDDATE_FACT_min': None,
  'MONTHS_BALANCE_min': -24.0,
  'DAYS_LAST_PHONE_CHANGE': -730.0,
  'CODE_GENDER_M': 1.0,
  'CODE_GENDER_F': 0.0,
  'DAYS_DECISION_min': -730

In [116]:
type(data_json)

list

On a bien une liste de dictionnaires au format json utilisable par l'api

On a deux façon de tester l'API :

1- en utilisant cURL en bash dans un terminal
```bash
curl -X POST "http://127.0.0.1:5000/predict" \
     -H "Content-Type: application/json" \
     --data-binary @data.json # fichier json
```

2- en utilisant la lib python request
```python
response = requests.post(
     url,
     json=data_json,
     headers={"Content-Type": "application/json"}
     )
```

In [13]:
# 4- on va envoyer une requête POST à l'API
response = requests.post(
    url, # adresse de l'API
    json=data_json, # données en format JSON
    headers={"Content-Type": "application/json"}) # type de contenu envoyé = JSON

In [14]:
# 5- on affiche la réponse de l'API
print("Status Code:", response.status_code) 
print("Response JSON:", response.json()) # response.json() si la réponse est en JSON

Status Code: 200
Response JSON: {'prediction': [0, 1, 0], 'probabilite_1': [0.22155905458029113, 0.5863958061210415, 0.03838863673327562], 'prediction_seuil': [0, 1, 0], 'top_features': [[['ELEVATORS_AVG', 1.2159773811869485], ['DAYS_CREDIT_ENDDATE_min', -0.6511893465777348], ['ELEVATORS_MEDI', -0.23732255623162168], ['APP_CREDIT_PERC_max', -0.1977917004155481], ['LIVINGAREA_AVG', -0.1779560036284692]], [['ELEVATORS_MEDI', 1.079754746127267], ['DAYS_CREDIT_ENDDATE_min', -0.3540463950196363], ['FLOORSMIN_AVG', -0.3148761269471951], ['NAME_INCOME_TYPE_Working', 0.27732742758827245], ['DEF_60_CNT_SOCIAL_CIRCLE', 0.222304849374506]], [['ELEVATORS_MEDI', -0.43041785944567423], ['ELEVATORS_AVG', -0.4153952916369237], ['DAYS_CREDIT_ENDDATE_min', -0.2862642840422188], ['DAYS_ENDDATE_FACT_mean', -0.1704404565710094], ['LIVINGAREA_MEDI', -0.16937771834797294]]]}


### 1-2 tests unitaires

Les tests unitaires sont des méthodes de test pour tester des unités de code source pour checker leur validité : fonction, module, class, method.
On teste depuis la plus petite unité jusqu'à la plus complexe.
les tests unitaires sont en général écrits dans un fichier de code différent.

Par défaut python a une lib de tests unitaires : unittest; on peut également installer la lib python pytest ou celle de fastAPI : testclient

### 1-3 Docker

Un conteneur docker qui comprend tous les éléments nécessaires pour exécuter une application, notamment les bibliothèques, les outils système, le code et le runtime. Il regroupe tout le code et les dépendances d’une application dans un format standard, qui permet une exécution rapide et de fiable dans l’ensemble des environnements informatiques. 

#### 1-4 Déploiement sur le cloud

On va désormais déployer notre conteneur sur le cloud en utilisant la lib Render.
Render va se charger de contruire l'image docker et de la lancer sur le serveur.
Il est relié au dépôt GitHub contenant le Dockerfile.